In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Encoders, Decoders, and Cross-Attention

Every Transformer stack we have built so far has been a **causal decoder** (that section): a triangular mask restricts token $i$ to attending to prefix positions $j \le i$, turning a single forward pass into $T$ parallel next-token predictions. Causal masking is essential when generating tokens one by one, since future tokens do not yet exist at inference time. When the *entire* input sequence is already available—for example, when embedding a document for retrieval, classifying tokens, encoding a source sentence for translation, or encoding an image patch grid (that section)—forbidding position $i$ from reading positions $j > i$ discards half of the surrounding context.

Depending on which tokens are available at the start of the computation and whether the input and output belong to the same sequence, the same `TransformerBlock` (that section) supports three canonical wirings (the figure):

1. **Encoder-only (bidirectional self-attention):** Every token attends to every token (`is_causal=False`), computing a context-aware representation of a complete input sequence in one pass (BERT [@Devlin.Chang.Lee.ea.2018], RoBERTa [@Liu.Ott.Goyal.ea.2019], ViT [@Dosovitskiy.Beyer.Kolesnikov.ea.2021]).
2. **Decoder-only (causal self-attention):** Each token attends only to itself and earlier positions (`is_causal=True`), supporting both prompt reading and autoregressive generation in a single homogeneous stack (GPT, Llama 3, Qwen3).
3. **Encoder--decoder (bidirectional encoder + causal decoder with cross-attention):** A bidirectional encoder maps a source sequence $\mathbf{X}_{\textrm{src}}$ to contextual representations $\mathbf{H}_{\textrm{enc}}$, and a causal decoder generates the target sequence $\mathbf{Y}_{\textrm{tgt}}$ one token at a time while querying $\mathbf{H}_{\textrm{enc}}$ through **cross-attention** (the original 2017 Transformer [@Vaswani.Shazeer.Parmar.ea.2017], T5 [@raffel2020exploring], Whisper [@radford2023whisper]).

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Encoder, Decoder, and Encoder--Decoder Architectures

![The three wirings and their attention patterns (query position runs downward, key position rightward; a filled cell means the query may attend to that key). Encoder-only: every token attends to every token. Decoder-only: each token attends only to itself and its past. Encoder--decoder: the source attends to itself bidirectionally, while each target token attends to the full source through cross-attention (light cells) and to its own past causally. The third panel hides one structural fact: its source rows and its target rows belong to two separate stacks, and every decoder block reads the output of the last encoder block.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-transformers-three-wirings.svg)

## Bidirectional Context and Masked-Token Prediction

### A Bidirectional Encoder

Removing the causal mask from `d2l.TransformerBlock` turns a causal stack into a **bidirectional Transformer encoder** (`TransformerEncoder`), mapping an input batch of token IDs (and an optional padding mask) to contextual representations of shape $(B, n, d)$:

In [ ]:
class TransformerEncoder(nnx.Module):
    """Bidirectional encoder: embeddings plus unmasked pre-norm blocks."""
    def __init__(self, vocab_size, num_hiddens=128, num_heads=4, num_blks=4,
                 max_len=64, *, rngs):
        init = nnx.initializers.normal(0.02)
        self.token_emb = nnx.Embed(vocab_size, num_hiddens,
                                   embedding_init=init, rngs=rngs)
        self.pos_emb = nnx.Embed(max_len, num_hiddens, embedding_init=init,
                                 rngs=rngs)
        self.blks = nnx.List([
            d2l.TransformerBlock(num_hiddens, num_heads, rngs=rngs)
            for _ in range(num_blks)])
        self.norm = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)

    def __call__(self, X, mask=None):
        H = self.token_emb(X) + self.pos_emb(jnp.arange(X.shape[1]))
        for blk in self.blks:
            H = blk(H, mask)  # by default no mask: all tokens see all
        return self.norm(H)

### The Masked Language Modeling (MLM) Objective

Why can a bidirectional stack not be trained with the next-token prediction objective (the equation) of that section? In a stack of $L \ge 2$ unmasked layers, position $t$ at block 1 attends to position $t+1$, and at block 2 reads its own target $x_{t+1}$ directly—collapsing next-token prediction to a trivial identity copy.

**Masked Language Modeling (MLM)**, introduced in **BERT** [@Devlin.Chang.Lee.ea.2018], replaces next-token prediction with a cloze task: randomly select a subset of positions $M \subset \{1, \dots, n\}$ (typically $p = 15\%$ of tokens), replace each $x_i$ ($i \in M$) with a dedicated `<mask>` token ID in the corrupted input $\tilde{\mathbf{x}}$, and minimize the cross-entropy of reconstructing the original tokens *only at the masked positions* $i \in M$:

$$
\mathcal{L}_{\textrm{MLM}}(\boldsymbol{\theta}) \;=\; -\frac{1}{|M|} \sum_{i \in M} \log P_{\boldsymbol{\theta}}(x_i \mid \tilde{\mathbf{x}}).
$$

We train a 4-block byte-level `MaskedLM` ($d = 128, H = 4, L = 4$) on the exact same WikiText-103 byte stream as in that section, reserving token ID `MASK = len(data.vocab)` ($256$) for `<mask>`:

In [ ]:
data = d2l.WikiText103(batch_size=64, num_steps=64)
MASK = len(data.vocab)  # id of the extra <mask> embedding row

def mask_tokens(X, key, p=0.15):
    mask = jax.random.uniform(key, X.shape) < p
    return jnp.where(mask, MASK, X), mask

class MaskedBytes:
    """The windows of data with masked bytes; labels -100 add no loss."""
    def __init__(self, data):
        self.data = data

    def train_dataloader(self):
        return self.masked(self.data.train_dataloader())

    def val_dataloader(self):
        return self.masked(self.data.val_dataloader())

    def masked(self, batches):
        key = d2l.get_key()
        for X, _ in batches:
            key, sub = jax.random.split(key)
            Xm, mask = mask_tokens(jnp.asarray(X), sub)
            yield Xm, jnp.where(mask, X, -100)

class MaskedLM(nnx.Module):
    """An encoder with an output head tied to its token embedding."""
    def __init__(self, encoder):
        self.encoder = encoder

    def __call__(self, X):
        return self.encoder.token_emb.attend(self.encoder(X))

model = MaskedLM(TransformerEncoder(len(data.vocab) + 1,
                                    rngs=nnx.Rngs(d2l.get_key())))
optimizer = nnx.Optimizer(model, optax.adamw(1e-3, weight_decay=0.0),
                          wrt=nnx.Param)
losses = d2l.train_steps(model, MaskedBytes(data), optimizer, 2000)
print('masked loss at step 500/1000/2000: ' + '/'.join(
    f'{sum(losses[k-50:k]) / 50:.2f}' for k in (500, 1000, 2000)))

### Loss by Available Context

In that section, a causal language model's per-position validation loss decreased monotonically from left ($t = 1$, no context) to right ($t = T$, full prefix context). What does `loss_by_position` look like for a bidirectional `MaskedLM` when we evaluate reconstruction loss at each window position $t \in \{0, \dots, n-1\}$?

In [ ]:
@nnx.jit
def scored_losses(model, X, Y):
    logits = model(X)
    losses = optax.softmax_cross_entropy_with_integer_labels(
        logits, jnp.maximum(Y, 0))
    return losses, logits.argmax(-1) == Y

def loss_by_position(model, batches):
    """Mean loss at each window position over the labels that are not -100,
    and the fraction of those labels predicted exactly."""
    model.eval()
    total = count = correct = 0
    for X, Y in batches:
        Y = jnp.asarray(Y)
        losses, hit = scored_losses(model, jnp.asarray(X), Y)
        scored = Y != -100
        total += (losses * scored).sum(0)
        count += scored.sum(0)
        correct += (hit & scored).sum()
    return total / count, float(correct / count.sum())

def show(losses):
    return ' '.join(f'{float(v):.2f}' for v in losses)

masked = MaskedBytes(data)
pos_loss, accuracy = loss_by_position(model, masked.val_dataloader())
interior = slice(16, 48)  # positions 16 to 47 of the 64
print(f'masked accuracy {accuracy:.2f}')
print(f'loss at positions 0-3: {show(pos_loss[:4])}; 60-63: '
      f'{show(pos_loss[-4:])}; interior mean {pos_loss[interior].mean():.2f}')

The bidirectional encoder traces a symmetric **bathtub (U-shaped) curve**: every interior position enjoys both left and right context (about 1.5–1.6 nats/byte), and loss rises only at the two window edges ($t \approx 0$ and $t \approx n-1$, about 2.4 nats/byte) where one side of the context is cut off by the window boundary. Evaluating `model` on a masked sentence shows how right context disambiguates missing interior bytes:

In [ ]:
snippet = 'Homarus gammarus , known as the European lobster or common '
ids = jnp.asarray(data.vocab.encode(snippet))[None]
for pos in (11, 22, 28):
    Xm = ids.at[0, pos].set(MASK)
    probs = jax.nn.softmax(model(Xm)[0, pos])
    top = jnp.argsort(probs)[::-1][:3]
    shown = snippet[:pos] + '_' + snippet[pos + 1:]
    print(f'{shown[:40]!r}... -> '
          + ', '.join(f'{data.vocab.decode([int(i)])!r} {probs[i]:.2f}'
                      for i in top))

## Cross-Attention in an Encoder--Decoder

### A Sequence-to-Sequence Task with Known Alignment

To study an **encoder--decoder** (the figure, right) on a clean sequence-to-sequence alignment problem, we train on string reversal over an alphabet of $V = 16$ letters (`a`--`p`) and length $T = 12$: given source string $\mathbf{x} = (x_1, \dots, x_T)$, the decoder receives $\textrm{dec\_in} = (\texttt{<bos>}, x_T, x_{T-1}, \dots, x_2)$ and predicts the reversed string $\mathbf{y} = (x_T, x_{T-1}, \dots, x_1)$:

In [ ]:
V, T = 16, 12   # 16 letters, strings of length 12
BOS = V         # decoder start token

def sample_batch(batch_size, key):
    src = jax.random.randint(key, (batch_size, T), 0, V)
    tgt = src[:, ::-1]
    dec_in = jnp.concatenate(
        [jnp.full((batch_size, 1), BOS), tgt[:, :-1]], 1)
    return src, dec_in, tgt

def to_str(ids):
    return ''.join(chr(ord('a') + int(i)) for i in ids)

src, dec_in, tgt = sample_batch(1, d2l.get_key())
print(f'source {to_str(src[0])!r} -> target {to_str(tgt[0])!r}')

### The Decoder Block and `EncoderDecoder`

A `DecoderBlock` inserts a **cross-attention** sublayer (`d2l.MultiHeadAttention`) between causal self-attention and the SwiGLU FFN. In cross-attention, **queries come from the decoder** ($\mathbf{Q} = \textrm{Norm}(\mathbf{Y})\mathbf{W}_q \in \mathbb{R}^{B \times T_{\textrm{dec}} \times d}$) while **keys and values come from the top of the encoder** ($\mathbf{K} = \mathbf{H}_{\textrm{enc}}\mathbf{W}_k \in \mathbb{R}^{B \times T_{\textrm{src}} \times d}$, $\mathbf{V} = \mathbf{H}_{\textrm{enc}}\mathbf{W}_v \in \mathbb{R}^{B \times T_{\textrm{src}} \times d}$). Because the entire encoded source $\mathbf{H}_{\textrm{enc}}$ exists before decoding begins, cross-attention uses **no causal mask**:

In [ ]:
class DecoderBlock(nnx.Module):
    """Pre-norm decoder block: causal self-attention, cross-attention,
    FFN."""
    def __init__(self, num_hiddens, num_heads, *, rngs):
        self.norm1 = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.norm2 = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.norm3 = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.self_attention = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                     rngs=rngs)
        self.cross_attention = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                      rngs=rngs)
        self.ffn = d2l.FeedForward(num_hiddens, rngs=rngs)

    def __call__(self, X, enc, mask=None, cross_mask=None):
        if mask is None:  # causal self-attention
            mask = d2l.causal_mask(X.shape[1])
        Y = self.norm1(X)
        X = X + self.self_attention(Y, Y, Y, mask)
        X = X + self.cross_attention(self.norm2(X), enc, enc, cross_mask)
        return X + self.ffn(self.norm3(X))

`EncoderDecoder` pairs a `TransformerEncoder` with a stack of `DecoderBlock` layers and a linear output head. Notice that `decode(dec_in, enc)` takes the precomputed encoder output `enc`, so during autoregressive generation the encoder runs **only once** on the source sequence while `decode` steps through the target tokens:

In [ ]:
class EncoderDecoder(nnx.Module):
    """A causal decoder cross-attending into a bidirectional encoder."""
    def __init__(self, vocab_size, num_hiddens=128, num_heads=4,
                 num_blks=1, max_len=64, *, rngs):
        init = nnx.initializers.normal(0.02)
        self.encoder = TransformerEncoder(vocab_size, num_hiddens,
                                          num_heads, num_blks, max_len,
                                          rngs=rngs)
        self.tgt_emb = nnx.Embed(vocab_size, num_hiddens,
                                 embedding_init=init, rngs=rngs)
        self.pos_emb = nnx.Embed(max_len, num_hiddens, embedding_init=init,
                                 rngs=rngs)
        self.blks = nnx.List([DecoderBlock(num_hiddens, num_heads,
                                           rngs=rngs)
                              for _ in range(num_blks)])
        self.norm = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.head = d2l.Linear(num_hiddens, vocab_size, use_bias=False,
                               rngs=rngs)

    def decode(self, dec_in, enc):
        H = self.tgt_emb(dec_in) + self.pos_emb(jnp.arange(dec_in.shape[1]))
        for blk in self.blks:
            H = blk(H, enc)
        return self.head(self.norm(H))

    def __call__(self, src, dec_in):
        return self.decode(dec_in, self.encoder(src))

### Training and Autoregressive Decoding

Training `EncoderDecoder` for 300 steps on random batches of 128 strings and running greedy autoregressive decoding (`reverse`) achieves near-perfect exact-string reversal:

In [ ]:
@nnx.jit
def s2s_step(model, optimizer, src, dec_in, tgt):
    def loss_fn(model):
        logits = model(src, dec_in)
        return optax.softmax_cross_entropy_with_integer_labels(
            logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1)).mean()
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

def train_reversal(model, num_steps=300):
    """Train on fresh batches of 128 strings; return the loss of each step."""
    optimizer = d2l.LMOptimizer(model, lr=1e-3, num_steps=num_steps,
                                warmup=0, weight_decay=0.0)
    key, losses = d2l.get_key(), []
    for step in range(num_steps):
        key, sub = jax.random.split(key)
        src, dec_in, tgt = sample_batch(128, sub)
        losses.append(s2s_step(model, optimizer, src, dec_in, tgt))
    return jax.device_get(losses)  # One transfer at the end

seq2seq = EncoderDecoder(V + 1, rngs=nnx.Rngs(d2l.get_key()))
losses = train_reversal(seq2seq)
print('loss at step 25/50/100/200/300: ' + '/'.join(
    f'{sum(losses[k-10:k]) / 10:.3f}' for k in (25, 50, 100, 200, 300)))

In [ ]:
@nnx.jit
def next_token(model, enc, buf, t):
    return model.decode(buf, enc)[:, t].argmax(-1)

def reverse(model, src):
    """Greedy decoding: the encoder runs once, the decoder once per token."""
    model.eval()
    enc = model.encoder(src)
    buf = jnp.full((len(src), T + 1), BOS)
    for t in range(T):
        buf = buf.at[:, t + 1].set(next_token(model, enc, buf, t))
    return buf[:, 1:]

src, _, tgt = sample_batch(1000, d2l.get_key())
pred = reverse(seq2seq, src)
print(f'exact match on 1000 fresh strings: '
      f'{(pred == tgt).all(-1).mean():.3f}')
print(f'source {to_str(src[0])!r} -> predicted {to_str(pred[0])!r}')

### Reading Cross-Attention Alignment

In string reversal, target step $t$ outputs source token $T - 1 - t$. Does the cross-attention weight matrix therefore have to concentrate on the anti-diagonal $(t, T - 1 - t)$? Let us compare our trained `EncoderDecoder`, with one encoder block and one decoder block (`with` encoder self-attention), against an otherwise identical model whose encoder has no blocks, only its embeddings (`without`, so encoder position $j$ can only hold information about source token $j$):

In [ ]:
def reversal_model(self_attention=True):
    model = EncoderDecoder(V + 1, rngs=nnx.Rngs(d2l.get_key()))
    if not self_attention:  # the encoder keeps embeddings and positions
        model.encoder = TransformerEncoder(V + 1, num_blks=0,
                                           rngs=nnx.Rngs(d2l.get_key()))
    train_reversal(model)
    return model

def alignment(model):
    """Exact match on 1000 fresh strings; the head-averaged cross-attention
    of 256 strings, and how often its largest weight in a row falls on the
    true source position."""
    src, _, tgt = sample_batch(1000, d2l.get_key())
    match = (reverse(model, src) == tgt).all(-1).mean()
    src, dec_in, _ = sample_batch(256, d2l.get_key())
    model(src, dec_in)
    maps = model.blks[0].cross_attention.attention_weights.mean(1)
    want = jnp.arange(T - 1, -1, -1)  # row t: column T-1-t
    return match, (maps.argmax(-1) == want).mean(), maps[0]

In [ ]:
encoders = {'with': [seq2seq] + [reversal_model() for _ in range(5)],
            'without': [reversal_model(False) for _ in range(6)]}
maps, titles = [], []
for name, models in encoders.items():
    runs = [alignment(m) for m in models]
    print(f'encoder {name} self-attention, six runs')
    print('  exact match:', ' '.join(f'{m:.3f}' for m, _, _ in runs))
    print('  rows on the true source position:',
          ' '.join(f'{100 * h:.0f}%' for _, h, _ in runs))
    _, hit, least = min(runs, key=lambda run: run[1])  # the least aligned
    maps.append(least)
    titles.append(f'{name}: {100 * hit:.0f}% of rows')
d2l.show_heatmaps(d2l.stack(maps)[None], xlabel='source position',
                  ylabel='target position', titles=titles, figsize=(6, 3),
                  cmap='Blues')

When the encoder has no self-attention (`without`), cross-attention *must* look directly at the anti-diagonal $(t, T - 1 - t)$ to retrieve source token $T - 1 - t$. When the encoder includes bidirectional self-attention (`with`), the encoder layers can already route or reorganize source information across positions before cross-attention reads $\mathbf{H}_{\textrm{enc}}$—a classic reminder [@Jain.Wallace.2019] that raw attention weights in a multi-layer network reflect inter-layer routing rather than a direct token-to-token attribution map.

## Cross-Attention as an Interface: Fixed-Size Latent Bottlenecks

Because cross-attention requires only that queries $\mathbf{Q} \in \mathbb{R}^{B \times M \times d}$ and key--values $\mathbf{K}, \mathbf{V} \in \mathbb{R}^{B \times N \times d}$ share feature width $d$, it can decouple internal processing length $M$ from raw input length $N$ (the figure).

![A latent bottleneck with input length $N$, $M$ learned latents, width $d$, and $L$ latent blocks. Every latent reads every input (shaded band); the read of one latent is drawn in full. One cross-attention read has mixing cost $O(MNd)$; each subsequent latent self-attention layer has mixing cost $O(M^2d)$, for $O(LM^2d)$ across the stack. These terms hold batch size, head count, and width fixed and omit projections and FFNs.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-transformers-latent-bottleneck.svg)

In **Perceiver** [@Jaegle.Gimeno.Brock.ea.2021] (and multimodal vision--language resamplers [@alayrac2022flamingo]), a learned array of $M \ll N$ latent query vectors $\mathbf{Z}_0 \in \mathbb{R}^{M \times d}$ cross-attends to an $N$-token input $\mathbf{X} \in \mathbb{R}^{B \times N \times d}$ in $O(M N d)$ operations, after which $L$ self-attention blocks process the compact latent array of length $M$ in $O(L M^2 d)$ operations instead of $O(L N^2 d)$:

In [ ]:
class PerceiverEncoder(nnx.Module):
    """M learned latents cross-attend into the input, then process among
    themselves."""
    def __init__(self, num_latents, num_hiddens, num_heads=4, num_blks=2,
                 *, rngs):
        self.latents = nnx.Param(
            0.02 * jax.random.normal(rngs.params(),
                                     (num_latents, num_hiddens)))
        self.norm_q = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)
        self.cross_attention = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                      rngs=rngs)
        self.blks = nnx.List([
            d2l.TransformerBlock(num_hiddens, num_heads, rngs=rngs)
            for _ in range(num_blks)])
        self.norm = nnx.RMSNorm(num_hiddens, epsilon=1e-5, rngs=rngs)

    def __call__(self, X):
        Z = jnp.broadcast_to(self.latents[...],
                             (X.shape[0],) + self.latents.shape)
        Z = Z + self.cross_attention(self.norm_q(Z), X, X)  # O(MN)
        for blk in self.blks:
            Z = blk(Z)                                       # O(M^2)
        return self.norm(Z)

## Choosing Among the Three Wirings

* **Encoder-only (`is_causal=False`)** gives every token full bidirectional context in a single pass, making it the standard backbone for dense text embeddings, reranking, token classification, and non-autoregressive vision backbones (that section).
* **Decoder-only (`is_causal=True`)** trains on $100\%$ of sequence tokens via next-token prediction (versus $15\%$ in MLM) and processes prompts and generated continuations in a single unified parameter stack and KV cache (that section), making it the dominant architecture for general-purpose language models.
* **Encoder--decoder (with cross-attention)** shines when the input and output have different modalities or lengths—such as speech spectrograms to text in Whisper [@radford2023whisper], or high-resolution visual/sensor arrays compressed through a fixed-size latent cross-attention interface (the figure).

## Summary

* The same `TransformerBlock` can be wired as a **bidirectional encoder** (`is_causal=False`), a **causal decoder** (`is_causal=True`), or a **cross-attending encoder--decoder** (the figure).
* Because unmasked multi-layer self-attention leaks future tokens under next-token prediction, bidirectional encoders are trained with **Masked Language Modeling (MLM)** (the equation), producing a symmetric U-shaped `loss_by_position` curve across the window.
* In an **encoder--decoder**, each `DecoderBlock` interleaves causal target self-attention with **cross-attention** (decoder queries attending to encoder keys and values) and a position-wise FFN; during generation, the encoder runs once on the source while the decoder steps autoregressively.
* **Cross-attention** with $M \ll N$ learned query vectors (`PerceiverEncoder`) compresses an $N$-token input into an $M$-token latent bottleneck in $O(M N d + L M^2 d)$ time (the figure).

## Exercises

1. **BERT's 80 / 10 / 10 masking recipe.** `mask_tokens` replaces every selected position ($15\%$ of bytes) with `<mask>`. Because `<mask>` never appears at fine-tuning or inference time, @Devlin.Chang.Lee.ea.2018 replace selected tokens with `<mask>` $80\%$ of the time, with a uniformly random vocabulary token $10\%$ of the time, and leave the original token unchanged $10\%$ of the time. Implement this 80 / 10 / 10 rule in `mask_tokens` and explain why keeping $10\%$ unmasked and $10\%$ random forces the encoder to maintain accurate representations at *unmasked* tokens as well as `<mask>` tokens.

2. **Padding masks on ragged sequence-to-sequence batches.** When a batch contains variable-length source strings of lengths $(s_1, \dots, s_B) \le S$ and target strings of lengths $(t_1, \dots, t_B) \le T_{\textrm{dec}}$, three boolean masks are needed: (a) a $(B, 1, S, S)$ encoder padding mask, (b) a $(B, 1, T_{\textrm{dec}}, T_{\textrm{dec}})$ decoder causal + padding mask, and (c) a $(B, 1, T_{\textrm{dec}}, S)$ rectangular cross-attention source-padding mask. Write a helper `make_seq2seq_masks(src_len, tgt_len, S, T_dec)` that constructs all three masks and verify that appending padded tokens to `src` in `EncoderDecoder` does not alter the non-padded decoder logits.

3. **Varying the MLM masking fraction $p$.** Train `MaskedLM` for 2,000 steps at masking fractions $p \in \{0.05, 0.15, 0.30, 0.50\}$ and evaluate all four models on the same $p = 0.15$ validation set `masked.val_dataloader()`. How does $p$ trade off the number of supervised gradient signals per batch against the amount of uncorrupted context available around each mask?

4. **Forward vs. backward direction of time in causal and bidirectional models.** A causal language model processes text left-to-right ($x_1 \to x_n$). Reverse every byte window in `WikiText103` so a causal model predicts right-to-left ($x_n \to x_1$), train for 1,000 steps, and compare its average per-byte validation loss and its `loss_by_position` curve against the left-to-right causal model and the bidirectional `MaskedLM`. Why is the total joint log-likelihood $\sum_{t=1}^n \log P(x_t \mid \textrm{past})$ theoretically invariant to time reversal, and where does the bidirectional model gain its advantage over both one-sided directions?

5. **Content-addressed vs. position-addressed cross-attention alignment.** In our string reversal experiment, every letter in a random string can repeat, so position is required to distinguish duplicate letters. Modify `sample_batch` so each length-12 string is a random permutation of 12 *distinct* letters, and train `EncoderDecoder` without positional embeddings in the encoder (`pos_emb.weight.zero_()`). Why does a 1-layer encoder without position embeddings fail on string reversal even when all 12 letters are distinct, whereas a sorting task (outputting the input letters in alphabetical order `a..p`) succeeds without any encoder positional embeddings?

6. **Perceiver IO for sequence-to-sequence reversal.** Perceiver IO [@Jaegle.Borgeaud.Alayrac.ea.2022] pairs a `PerceiverEncoder` ($M$ learned latent queries attending to the $N$-token input) with a cross-attention **decoder** in which $N_{\textrm{out}}$ target position embeddings $\mathbf{Q}_{\textrm{out}} \in \mathbb{R}^{N_{\textrm{out}} \times d}$ cross-attend to the $M$ latent vectors $\mathbf{Z}^{(L)} \in \mathbb{R}^{B \times M \times d}$. Build a `PerceiverAutoencoder` that encodes a 12-letter string into $M \in \{2, 4, 8, 12\}$ latents ($d = 128$) and decodes the reversed string in one parallel cross-attention readout pass. How does reversal accuracy vary with the bottleneck size $M$?

[Discussions](https://d2l.discourse.group/t/36777)